In [1]:
import pandas as pd
import numpy as np

mock_data = {
    'Account': ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8'],
    'DeviceUniqueIdentifier': [
        'Dev_1',          
        'Dev_2', 'Dev_2', 
        'Dev_3',          
        'Dev_4',          
        'Dev_5', 'Dev_5', 
        'Dev_6'          
    ],
    'SystemMemorySize': [1024, 1535, 1535, 2048, 2500, 4096, 4096, 8192],
    'AdRevenue':        [1.5,   0.0,   2.5,   np.nan, 3.0,   1.0,   4.0,   np.nan],
    'BillingRevenue':   [0.0,   5.0,   np.nan, 10.0,  0.0,   15.0,  5.0,   np.nan]
}

df_raw = pd.DataFrame(mock_data)
df_raw.to_csv('raw_data.csv', index=False)
print("Файл raw_data.csv успешно создан!\n")

data = pd.read_csv('raw_data.csv')

device_df = data.groupby('DeviceUniqueIdentifier', as_index=False).agg(
    SystemMemorySize=('SystemMemorySize', 'first'),
    ad_rev=('AdRevenue', 'sum'),
    billing_rev=('BillingRevenue', 'sum')
)

device_df['total_revenue'] = device_df['ad_rev'] + device_df['billing_rev']

bins = [-1, 1535, 2048, 3095, np.inf]
labels = ['0–1535 MB', '1536–2048 MB', '2049–3095 MB', '3096+ MB']
device_df['ram_group'] = pd.cut(device_df['SystemMemorySize'], bins=bins, labels=labels)

result = (
    device_df.groupby('ram_group', observed=False)
    .agg(
        devices_count=('DeviceUniqueIdentifier', 'count'),
        total_revenue=('total_revenue', 'sum'),
        avg_revenue_per_device=('total_revenue', 'mean')
    )
    .reset_index()
)

print("Результат работы алгоритма:")
print(result)

Файл raw_data.csv успешно создан!

Результат работы алгоритма:
      ram_group  devices_count  total_revenue  avg_revenue_per_device
0     0–1535 MB              2            9.0                     4.5
1  1536–2048 MB              1           10.0                    10.0
2  2049–3095 MB              1            3.0                     3.0
3      3096+ MB              2           25.0                    12.5
